# AdaptLab: treino QLoRA (NF4) e avaliação em GPU

Notebook para **Colab ou Kaggle com GPU CUDA** (T4 ou superior). Não corre no PC local (sem CUDA).

Passos: verificar GPU, instalar versões fixadas, correr o treino, correr a avaliação, exportar os resultados.

Antes de começar: carregar a pasta `adaptlab/` (ou um zip dela) para `/content/adaptlab` (Colab) ou `/kaggle/working/adaptlab` (Kaggle). Não incluir `.venv/`, `runs/` nem pesos.

Versões fixadas (verificadas no PyPI em 2026-11-21):

| Pacote | Versão |
|---|---|
| transformers | 5.23.0 |
| trl | 1.18.2 |
| peft | 0.22.0 |
| datasets | 5.2.0 |
| accelerate | 1.16.0 |
| bitsandbytes | 0.50.2 (só CUDA) |
| pydantic | 2.14.4 |
| pyyaml | 6.0.3 |
| scikit-learn | 1.9.1 |
| numpy | 2.5.5 |

O torch já vem instalado no Colab/Kaggle com CUDA; não o reinstalamos para não partir o ambiente.

In [ ]:
import subprocess

result = subprocess.run(["nvidia-smi"], capture_output=True, text=True)
print(result.stdout or result.stderr)

import torch
assert torch.cuda.is_available(), "Sem GPU CUDA: activa o acelerador GPU no Colab/Kaggle."
print("GPU:", torch.cuda.get_device_name(0), "| bf16:", torch.cuda.is_bf16_supported())

In [ ]:
import subprocess

PINNED = [
    "transformers==5.23.0",
    "trl==1.18.2",
    "peft==0.22.0",
    "datasets==5.2.0",
    "accelerate==1.16.0",
    "bitsandbytes==0.50.2",
    "pydantic==2.14.4",
    "pyyaml==6.0.3",
    "scikit-learn==1.9.1",
    "numpy==2.5.5",
]
subprocess.run(["pip", "install", "-q", *PINNED], check=True)
print("instalado")

In [ ]:
import os

CANDIDATES = ["/content/adaptlab", "/kaggle/working/adaptlab"]
PROJECT = next((p for p in CANDIDATES if os.path.isfile(os.path.join(p, "scripts", "train.py"))), None)
if PROJECT is None:
    raise FileNotFoundError("Carrega a pasta adaptlab/ para /content/adaptlab antes de continuar.")
os.chdir(PROJECT)
print("projecto:", PROJECT)

In [ ]:
import subprocess

# Verifica os dados antes de gastar GPU: falha se houver fugas ou labels inválidos.
subprocess.run(["python", "scripts/prepare_data.py", "--check"], check=True)
# Treino QLoRA (NF4). Config: configs/train_gpu.yaml (igual a train.yaml, quantization: nf4).
subprocess.run(["python", "scripts/train.py", "--config", "configs/train_gpu.yaml"], check=True)

In [ ]:
import glob
import subprocess

# Avaliação final: baselines (zero-shot, few-shot, regras) e LoRA no mesmo teste.
# Usa a última run de treino. O teste é lido aqui pela primeira vez (verificação de hash incluída).
latest = sorted(glob.glob("runs/train-*"))[-1]
subprocess.run(["python", "scripts/evaluate.py", "--run-dir", latest, "--split", "test", "--final"], check=True)

In [ ]:
import glob
import os
import shutil

# Exporta só o adapter, a configuração, as métricas e os relatórios (nunca os pesos do modelo base).
latest = sorted(glob.glob("runs/train-*"))[-1]
archive = shutil.make_archive("/content/adaptlab_results" if os.path.isdir("/content") else "/kaggle/working/adaptlab_results", "zip", root_dir=latest)
print("exportado:", archive)